In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch

In [4]:
# !pip install gpytorch

In [17]:
import gpytorch
from gpytorch.models.deep_gps import DeepGP, DeepGPLayer
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution
from gpytorch.variational import MeanFieldVariationalDistribution
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import VariationalELBO
from gpytorch.mlls import VariationalELBO, DeepApproximateMLL

In [6]:
df = pd.read_csv(r"C:\Users\cecil\Cript_Anomalies\BTCUSDT_20180101_20260112.csv", delimiter=";", skiprows=0)
display(df)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,n_trades,taker_buy_base,...,ignore,symbol,interval,log_return,volatility_20,range_hl,trades_per_volume,buy_ratio,z_return,anomaly_simple
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,0,BTCUSDT,15m,NaN,NaN,0.023013,12.716799,0.511480,NaN,False
1,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,0,BTCUSDT,15m,-0.002587,NaN,0.011000,14.887438,0.485919,-0.691986,False
2,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,0,BTCUSDT,15m,-0.003757,NaN,0.007064,12.515012,0.547036,-1.004095,False
3,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,0,BTCUSDT,15m,0.004341,NaN,0.017849,8.433327,0.521511,1.156086,False
4,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,0,BTCUSDT,15m,-0.006182,NaN,0.012526,12.379798,0.472275,-1.650855,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281126,2026-01-12 22:30:00+00:00,91269.52,91402.81,91203.69,91339.90,86.681550,2026-01-12 22:44:59.999000+00:00,7.912440e+06,28813,50.482400,...,0,BTCUSDT,15m,0.000771,0.001837,0.002182,332.400609,0.582389,0.203804,False
281127,2026-01-12 22:45:00+00:00,91339.89,91339.90,91214.00,91244.99,60.451790,2026-01-12 22:59:59.999000+00:00,5.517664e+06,19198,18.105290,...,0,BTCUSDT,15m,-0.001040,0.001788,0.001378,317.575377,0.299500,-0.279123,False
281128,2026-01-12 23:00:00+00:00,91244.99,91280.99,91059.88,91160.07,180.361250,2026-01-12 23:14:59.999000+00:00,1.643558e+07,26239,91.014310,...,0,BTCUSDT,15m,-0.000931,0.001721,0.002423,145.480251,0.504622,-0.250178,False
281129,2026-01-12 23:15:00+00:00,91160.07,91282.39,91135.58,91269.34,31.328540,2026-01-12 23:29:59.999000+00:00,2.857620e+06,9456,19.653920,...,0,BTCUSDT,15m,0.001198,0.001643,0.001610,301.833408,0.627349,0.317736,False


In [23]:
class DGPHiddenLayer(DeepGPLayer):
    def __init__(self, input_dims, output_dims, num_inducing=256):
        # Inducing points: [M, input_dims]
        inducing_points = torch.randn(num_inducing, input_dims)

        variational_distribution = CholeskyVariationalDistribution(num_inducing_points=num_inducing)
        variational_strategy = VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True
        )
        super().__init__(variational_strategy, input_dims, output_dims)

        self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([output_dims]))
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(batch_shape=torch.Size([output_dims])),
            batch_shape=torch.Size([output_dims])
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class TwoLayerDGP(DeepGP):
    def __init__(self, input_dims, hidden_dims=2, num_inducing=256):
        super().__init__()

        self.hidden_layer = DGPHiddenLayer(
            input_dims=input_dims,
            output_dims=hidden_dims,
            num_inducing=num_inducing
        )
        self.last_layer = DGPHiddenLayer(
            input_dims=hidden_dims,
            output_dims=1,
            num_inducing=num_inducing
        )

        self.likelihood = GaussianLikelihood()

    def forward(self, x):
        hidden_rep = self.hidden_layer(x)
        output = self.last_layer(hidden_rep)
        return output
    
    # def forward(self, x):
    #     h = self.hidden_layer(x)
    #     out = self.last_layer(h)
    #     return out


In [24]:

def fit_deep_gp_anomaly(
    df: pd.DataFrame,
    x_cols,
    y_col: str = "log_return",
    contamination: float = 0.01,
    train_n: int = 60000,          # subsample para entrenar (ajusta)
    num_inducing: int = 256,
    hidden_dims: int = 2,
    epochs: int = 8,
    batch_size: int = 2048,
    lr: float = 0.01,
    device: str | None = None,
    random_state: int = 42
):
    """
    Entrena DGP (2 capas) para predecir y_col a partir de x_cols.
    Score = NLPD (negative log predictive density) aproximado.
    Devuelve:
      df_out con dgp_nlpd, anomaly_dgp
      model, scalers, threshold
    """

    rng = np.random.default_rng(random_state)
    df_out = df.copy()

    # --- limpiar / alinear ---
    cols = list(x_cols) + [y_col]
    d = df_out[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    idx_all = d.index

    # --- escalar X y (opcional, ayuda) ---
    sx = StandardScaler()
    X = sx.fit_transform(d[x_cols].values).astype(np.float32)

    sy = StandardScaler()
    y = sy.fit_transform(d[[y_col]].values).astype(np.float32).reshape(-1)

    # --- elegir train subset ---
    n = len(d)
    if train_n is not None and n > train_n:
        train_idx = rng.choice(n, size=train_n, replace=False)
    else:
        train_idx = np.arange(n)

    X_train = X[train_idx]
    y_train = y[train_idx]

    # --- torch tensors ---
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)

    model = TwoLayerDGP(input_dims=X.shape[1], hidden_dims=hidden_dims, num_inducing=num_inducing).to(device)
    likelihood = model.likelihood.to(device)

    xb, yb = next(iter(train_loader))
    out = model(xb)
    print("out batch_shape:", out.batch_shape)
    print("out event_shape:", out.event_shape)
    # model.train()
    # likelihood.train()

    # optimizer = torch.optim.Adam([
    #     {"params": model.parameters()},
    #     {"params": likelihood.parameters()}
    # ], lr=lr)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ELBO
    #mll = VariationalELBO(likelihood, model, num_data=X_train_t.shape[0])
    mll = DeepApproximateMLL(
        VariationalELBO(likelihood, model, num_data=X_train_t.shape[0])
    )

    # mini-batches
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # for ep in range(1, epochs + 1):
    #     losses = []
    #     for xb, yb in train_loader:
    #         optimizer.zero_grad()
    #         output = model(xb)
    #         loss = -mll(output, yb)
    #         loss.backward()
    #         optimizer.step()
    #         losses.append(loss.item())
    #     print(f"Epoch {ep}/{epochs} - Loss: {float(np.mean(losses)):.4f}")

    for ep in range(1, epochs + 1):
        losses = []
        for xb, yb in train_loader:
            xb = xb.reshape(xb.shape[0], -1)      # [B, D]
            yb = yb.reshape(-1)                   # [B]

            optimizer.zero_grad()

            with gpytorch.settings.num_likelihood_samples(10):
                output = model(xb)
                loss = -mll(output, yb)

            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        print(f"Epoch {ep}/{epochs} - Loss: {float(np.mean(losses)):.4f}")

    # --- scoring en todo el dataset limpio ---
    model.eval()
    likelihood.eval()

    X_all_t = torch.from_numpy(X).to(device)

    # NLPD aproximado: -log p(y | x)
    # Usamos predictive distribution de likelihood(model(x))
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_dist = likelihood(model(X_all_t))  # Normal approx
        # log_prob por punto
        logp = pred_dist.log_prob(torch.from_numpy(y).to(device))
        nlpd = (-logp).detach().cpu().numpy()

    # threshold por percentil (contamination)
    thr = float(np.quantile(nlpd, 1 - contamination))

    # reinyectar a df original
    df_out["dgp_nlpd"] = np.nan
    df_out["anomaly_dgp"] = np.nan
    df_out.loc[idx_all, "dgp_nlpd"] = nlpd
    df_out.loc[idx_all, "anomaly_dgp"] = (nlpd >= thr)

    return df_out, model, (sx, sy), thr

In [12]:
def compute_anomaly_metrics(
    df: pd.DataFrame,
    anomaly_col: str,
    score_col: str,
    baseline_col: str = "anomaly_simple"
):
    """
    Versión genérica de tus métricas (igual estilo que compute_ee_metrics),
    para cualquier modelo con:
      - anomaly_col: columna booleana (True anomalía)
      - score_col: score (mayor = más anómalo)
    """

    df_eval = df.dropna(subset=[anomaly_col])

    total = len(df_eval)
    anomalies = df_eval[anomaly_col].astype(bool).sum()
    rate = anomalies / total if total else 0

    metrics = {
        "total_points": total,
        "total_anomalies": int(anomalies),
        "anomaly_rate": float(rate),
        "score_mean": float(df_eval[score_col].mean()),
        "score_std": float(df_eval[score_col].std()),
        "score_p95": float(df_eval[score_col].quantile(0.95)),
        "score_p99": float(df_eval[score_col].quantile(0.99)),
        "score_max": float(df_eval[score_col].max()),
    }

    if baseline_col in df.columns:
        overlap = (
            df_eval[anomaly_col].astype(bool) &
            df_eval[baseline_col].astype(bool)
        ).sum()
        metrics["baseline_overlap"] = int(overlap)

    return metrics

In [9]:
features = [
    "log_return",
    "volatility_20",
    "range_hl",
    "trades_per_volume",
    "buy_ratio"
]

In [10]:
df_clean = df.copy()

# 1) quitar inf
df_clean[features] = df_clean[features].replace([np.inf, -np.inf], np.nan)

# 2) eliminar filas con NA en features
df_clean = df_clean.dropna(subset=features).reset_index(drop=True)

In [25]:
df_dgp, dgp_model, scalers, thr = fit_deep_gp_anomaly(
    df=df_clean,             # si ya limpiaste NaN
    x_cols=features,
    y_col="log_return",
    contamination=0.01,
    train_n=60000,
    epochs=8,
    batch_size=2048
)

UnboundLocalError: cannot access local variable 'train_loader' where it is not associated with a value

In [ ]:
metrics_dgp = compute_anomaly_metrics(
    df_dgp,
    anomaly_col="anomaly_dgp",
    score_col="dgp_nlpd"
)
metrics_dgp